# Discrete flow matching language model (`variant: flow_matching`)

<img src="../../assets/explainers/flow_matching.gif" width="100%">

**Training**: the probability path - as $t$ goes 0 → 1 each character turns from a mask into data with probability $\kappa(t)=t$ (one coupled draw per character, so the path is continuous); training picks a $t$ and learns to fill the masks. **Generation**: Euler steps $t: 0 \to 1$; each step the model guesses every `?` and each jumps to its guess with probability $h\,\kappa'/(1-\kappa)$.

## Intuition

Flow matching defines a path of distributions $p_t$ from a simple source ($t=0$) to the data ($t=1$) and
learns the "velocity" that moves samples along it. For tokens with a **masking** source, the velocity is
"jump from `[MASK]` to the predicted token", so the model is again a denoiser $p_\theta(x_1 \mid x_t)$
(Campbell et al. 2024; Gat et al. 2024).

## Math

$$p_t(x^i \mid x_1^i) = \kappa(t)\,\delta_{x_1^i} + (1-\kappa(t))\,\delta_{[\mathrm{M}]},\qquad
  \kappa(0) = 0,\ \kappa(1) = 1.$$

The generating rate of the mixture path moves a masked token to $x_1$ at rate
$u_t = \frac{\kappa'(t)}{1-\kappa(t)}\, p_\theta(x_1 \mid x_t)$. Training minimises the masked CE weighted by
the same rate; with $\kappa(t) = t$ it is $1/(1-t)$:

$$\mathcal{L} = \mathbb{E}_{t \sim U[0,1)}\Big[\frac{1}{1-t}\sum_{i:\,x_t^i=[\mathrm{M}]} -\log p_\theta(x_1^i \mid x_t, t)\Big].$$

**Euler step** $t \to t+h$: each masked token jumps w.p. $h\,\kappa'(t)/(1-\kappa(t))$ (here $h/(1-t)$);
the last step has probability 1.

**Relation to masked diffusion.** With linear $\kappa$, flow time $t$ is diffusion time $1-t$:
the two training objectives and samplers coincide up to that reparameterisation. Flow matching makes the
path ($\kappa$) and the step rule explicit design choices - see `07_block_flow_matching` for a cosine path.

## In the code

| File | What |
|---|---|
| `models_flow_matching/model.py` | `FlowMatchingLM`, `flow_matching_loss`, `sample_flow_matching` |
| `models_flow_matching/vector_field.py` | `mask_prob` (the linear path) |
| `models_flow_matching/metrics.py` | `masked_accuracy` |

In [ ]:
import sys
from pathlib import Path

# Run from anywhere inside the repo: put src/ on the path if hale_llm is not installed.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import HTML, display

import hale_llm.models  # noqa: F401  - registers every paradigm
from hale_llm.core.registry import get_loss, get_sampler, get_variant
from hale_llm.dataloader.collate import get_collate
from hale_llm.visualization.explainers import CORPUS, get_explainer

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
print("training corpus:")
for sentence in CORPUS:
    print("  ", repr(sentence))

## Train a toy model

Bidirectional attention + time conditioning. The explainer's recipe trains a ~100k-parameter model on the four corpus sentences
in a few seconds on a CPU - small enough to inspect every number, trained enough to show real
behaviour.

In [ ]:
explainer = get_explainer("flow_matching")
run = explainer.train()
tok, model, cfg = run.tokenizer, run.model, run.cfg
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("config:", {k: v for k, v in cfg.model.model_dump().items()
                  if k in ("arch", "d_model", "n_layers", "attn_type", "use_time_cond",
                           "block_size", "n_mtp_heads", "fm_schedule")})

fig, ax = plt.subplots(figsize=(6.5, 2.4))
ax.semilogy(run.losses, color="#264653")
ax.set(xlabel="step", ylabel="training loss", title="toy training")
plt.show()

## One coupled path from noise to data

In [ ]:
torch.manual_seed(7)
u = torch.rand(48)
for t in (0.0, 0.2, 0.4, 0.6, 0.8, 1.0):
    print(f"t={t:.1f}", "".join(c if u[i] < t else "_" for i, c in enumerate(run.text)))

## Rate = loss weight

In [ ]:
import numpy as np

ts = np.linspace(0, 0.98, 99)
fig, ax = plt.subplots(1, 2, figsize=(9, 2.6))
ax[0].plot(ts, ts, color="#264653"); ax[0].set(title="κ(t) = t", xlabel="t")
ax[1].plot(ts, 1 / (1 - ts), color="#F4A261"); ax[1].set(title="rate κ′/(1−κ) = 1/(1−t)", xlabel="t", yscale="log")
plt.show()

In [ ]:
def validity(steps_list, seeds=24, temperature=1.0):
    """Share of samples that are exactly one of the training sentences, per sampling-step budget."""
    prompt = run.ids[:, :8]
    sampler = get_sampler(cfg.variant)
    out = {}
    for steps in steps_list:
        ok = 0
        for seed in range(seeds):
            torch.manual_seed(seed)
            ids = sampler(model, tok, prompt, max_new_tokens=40, temperature=temperature, sampling_steps=steps)
            ok += tok.decode(ids[0]) in CORPUS
        out[steps] = ok / seeds
    return out

In [ ]:
scores = validity([2, 4, 8, 16, 40])
print("valid share per Euler step budget:", scores)

In [ ]:
trace = explainer.trace(run, seed=0)
assert torch.equal(trace.ids, explainer.reference(run, seed=0))
for s in trace.steps:
    canvas = "".join("_" if v == tok.mask_token_id else tok.token(v) for v in s.x)
    print(f"t {s.t_from:.3f}→{s.t_to:.3f}  p_jump {s.p_reveal:.2f}  +{len(s.revealed):2d}  {canvas}")

In [ ]:
# Re-render the animation from this notebook's model. The committed copy shown at the top is
# assets/explainers/flow_matching.gif (regenerate all of them with `hale-llm-explain`).
from PIL import Image

out = ROOT / "notebooks" / "models" / "outputs" / "flow_matching.gif"  # git-ignored
explainer.build(out, run=run)
with Image.open(out) as gif:
    print(out.relative_to(ROOT), f"{gif.n_frames} frames, {gif.size[0]}x{gif.size[1]},",
          f"{out.stat().st_size / 1024:.0f} KB")
# In Jupyter: display(HTML('<img src="outputs/flow_matching.gif" width="100%">'))

## Takeaways

* The masking flow is a denoiser trained with a rate-weighted CE; sampling integrates $t: 0 \to 1$.
* Time runs the other way from masked diffusion, and the path $\kappa$ is a free design choice.
* Same parallel-decoding trade-off: fewer Euler steps → more simultaneous jumps → more mixing.

**Exercise.** Implement a cosine $\kappa$ for this variant using `MaskingPath` from
`models_block_flow_matching/path.py` and compare `validity` at 4 steps.